# Pipeline de Revisión Científica — Fases 1 a 3

Este notebook recorre el pipeline completo de búsqueda, procesamiento y búsqueda semántica de literatura científica.

```
┌──────────────────────────────────────────────────────────────────┐
│  FASE 1          FASE 2              FASE 3                      │
│                                                                  │
│  Búsqueda   →   Embeddings   →   RAG Pipeline                   │
│  (APIs)         (SentenceT.)      PDF → Chunks → FAISS → Search  │
└──────────────────────────────────────────────────────────────────┘
```

**Tema del demo:** *Lutjanus peru* — pesquería del pargo del Pacífico en el Golfo de California.

## 0. Configuración

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

# Asegurar que la raíz del proyecto está en el path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Directorio raíz: {ROOT}')

# Verificar dependencias
deps = {}
for pkg, label in [('numpy','numpy'), ('faiss','faiss-cpu'),
                   ('pdfplumber','pdfplumber'), ('sentence_transformers','sentence-transformers'),
                   ('requests','requests')]:
    try:
        __import__(pkg)
        deps[label] = '✅'
    except ImportError:
        deps[label] = '❌ pip install ' + label

for k, v in deps.items():
    print(f'  {k:25s} {v}')

---
## FASE 1 — Búsqueda de Artículos Científicos

El módulo `scientific_search` unifica la búsqueda en tres fuentes académicas simultáneamente:

| Fuente | Cobertura | Acceso |
|--------|-----------|--------|
| **Crossref** | 130M+ artículos, todas las disciplinas | Abierto |
| **PubMed** | Ciencias de la salud y biología | Abierto |
| **arXiv** | Preprints de ciencias exactas | Abierto |
| **Scopus** | Base curada con índice de citas | API key |

### Arquitectura interna
```
ScientificArticleSearcher
    ├── CrossrefAdapter   → API REST de Crossref
    ├── PubMedAdapter     → E-utilities de NCBI
    └── ArxivAdapter      → API de arXiv
         ↓
    deduplicate() + relevance_filter()
         ↓
    SearchResult(articles=[Article, ...])
```

Cada `Article` tiene: `title`, `authors`, `year`, `doi`, `abstract`, `keywords`, `journal`, `source`.

In [ ]:
from scientific_search import ScientificArticleSearcher
from scientific_search.models import Article, SearchResult

QUERY = "Lutjanus peru fisheries Gulf of California"
MAX_RESULTS = 10   # por fuente
YEAR_START  = 2000

searcher = ScientificArticleSearcher(
    sources=['crossref', 'pubmed', 'arxiv'],
    output_directory=ROOT / 'outputs',
    verbose=False,
)

print(f'Buscando: "{QUERY}"')
print(f'Fuentes: crossref, pubmed, arxiv | Desde: {YEAR_START}\n')

result: SearchResult = searcher.search(
    query=QUERY,
    max_results=MAX_RESULTS,
    year_start=YEAR_START,
    min_relevance=0.3,
)

print(f'Artículos encontrados (únicos y relevantes): {len(result.articles)}')
print(f'Fuentes consultadas: {", ".join(result.sources_queried)}')
if result.errors:
    print(f'Errores: {result.errors}')

In [ ]:
# Mostrar tabla de resultados
print(f'{'#':<4} {'Año':<6} {'Fuente':<10} {'Título'}') 
print('-' * 90)
for i, a in enumerate(result.articles, 1):
    title_short = a.title[:65] + '...' if len(a.title) > 65 else a.title
    print(f'{i:<4} {str(a.year or "-"):<6} {a.source:<10} {title_short}')

In [ ]:
# Ver detalle del primer artículo
if result.articles:
    a = result.articles[0]
    print(f'TÍTULO    : {a.title}')
    print(f'AUTORES   : {", ".join(a.authors[:4])}{" +más" if len(a.authors)>4 else ""}')
    print(f'AÑO       : {a.year}')
    print(f'JOURNAL   : {a.journal or "N/A"}')
    print(f'DOI       : {a.doi or "N/A"}')
    print(f'FUENTE    : {a.source}')
    print(f'\nRESUMEN:\n{a.abstract[:500]}...' if a.abstract and len(a.abstract) > 500
          else f'\nRESUMEN:\n{a.abstract or "Sin resumen"}')

In [ ]:
# Guardar resultados en CSV
saved = searcher.registry.save_search_result(result)
print('Archivos guardados:')
for tipo, ruta in saved.items():
    print(f'  {tipo:10s} → {ruta}')

---
## FASE 2 — Generación de Embeddings

Un **embedding** es una representación numérica densa del significado semántico de un texto. Textos similares producen vectores con alta similitud coseno.

```
"Pacific red snapper population"  →  [0.12, -0.34, 0.87, ...]  (384 dims)
"Lutjanus peru Gulf of California" →  [0.11, -0.36, 0.89, ...]  (similar)
"Climate change temperature"       →  [-0.45, 0.23, -0.12, ...]  (distinto)
```

### Modelos disponibles

| Proveedor | Modelo | Dims | Requisito |
|-----------|--------|------|-----------|
| **Local** | all-MiniLM-L6-v2 | 384 | `sentence-transformers` |
| Local | all-mpnet-base-v2 | 768 | `sentence-transformers` |
| **OpenAI** | text-embedding-3-small | 512 | API key |
| OpenAI | text-embedding-3-large | 3072 | API key |

La clase base `EmbeddingGenerator` define la interfaz; `get_embedding_generator()` actúa como factory.

In [ ]:
from pipeline.embeddings.embedding_generator import get_embedding_generator
import numpy as np

# Cargar modelo local (no requiere API key, funciona sin internet tras descarga)
gen = get_embedding_generator(provider='local', verbose=False)

print(f'Modelo     : {gen.get_model_name()}')
print(f'Dimensión  : {gen.get_dimension()} dims')
print(f'Tipo       : {gen.__class__.__name__}')

In [ ]:
# Generar embeddings para los abstracts encontrados en la búsqueda
# Combinamos título + abstract para maximizar la información semántica

articles_with_abstract = [a for a in result.articles if a.abstract]

texts = [
    f"{a.title}. {a.abstract}"
    for a in articles_with_abstract
]

print(f'Generando embeddings para {len(texts)} artículos...')
embeddings = gen.batch_generate(texts, batch_size=16, show_progress=True)

print(f'\nShape del resultado : {embeddings.shape}  (artículos × dims)')
print(f'Dtype               : {embeddings.dtype}')
print(f'Norma del primer vec: {np.linalg.norm(embeddings[0]):.4f}  (≈1.0 si normalizado)')

In [ ]:
# Matriz de similitud coseno entre artículos
# Valor cercano a 1.0 → artículos muy similares semánticamente

from pipeline.embeddings.embedding_generator import EmbeddingGenerator

n = len(embeddings)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = EmbeddingGenerator.cosine_similarity(
            embeddings[i], embeddings[j]
        )

print('MATRIZ DE SIMILITUD COSENO (artículos con abstract)')
print('(1.0 = idéntico, 0.0 = sin relación)\n')

# Encabezado
labels = [f'A{i+1}' for i in range(n)]
print(f'     {"  ".join(f"{l:>5}" for l in labels)}')
print('    ' + '-' * (7 * n))
for i, row in enumerate(sim_matrix):
    vals = '  '.join(f'{v:5.2f}' for v in row)
    print(f'{labels[i]:>4} {vals}')

print()
for i, a in enumerate(articles_with_abstract):
    title_short = a.title[:70] + '...' if len(a.title) > 70 else a.title
    print(f'  A{i+1}: {title_short}')

In [ ]:
# Encontrar el par de artículos más similares
max_sim = -1
pair = (0, 1)
for i in range(n):
    for j in range(i+1, n):
        if sim_matrix[i, j] > max_sim:
            max_sim = sim_matrix[i, j]
            pair = (i, j)

a1 = articles_with_abstract[pair[0]]
a2 = articles_with_abstract[pair[1]]

print(f'Par más similar (score={max_sim:.3f}):')
print(f'  A{pair[0]+1}: {a1.title}')
print(f'  A{pair[1]+1}: {a2.title}')

---
## FASE 3 — RAG Pipeline

**RAG** (Retrieval-Augmented Generation) permite hacer preguntas sobre un corpus de documentos.
La Fase 3 construye el índice de recuperación:

```
PDF
 │
 ├─ PdfPlumberExtractor  →  [(página, texto), ...]      extracción
 │
 ├─ TextChunker          →  [ChunkData, ...]             fragmentación
 │       chunk_size=2000 chars, overlap=200 chars
 │
 ├─ EmbeddingGenerator   →  np.ndarray (N × 384)         vectorización
 │
 └─ VectorDBManager       →  FAISS FlatIP index           almacenamiento
         index.faiss
         metadata_store.json
         index_config.json
```

### ¿Por qué fragmentar en chunks?

Los modelos de embedding tienen un límite de tokens (~512). Un paper de 20 páginas tiene ~40.000 palabras.  
Al fragmentar en chunks de ~500 tokens con solapamiento, cada chunk es autónomo semánticamente  
y recuperable independientemente.

### 3.1 — TextChunker: fragmentación de texto

In [ ]:
from pipeline.rag import TextChunker, ChunkData

chunker = TextChunker(
    chunk_size=2000,   # ~512 tokens, tamaño ideal para all-MiniLM-L6-v2
    overlap=200,       # 10% de solapamiento: preserva contexto entre chunks
    min_chunk_size=100,
    verbose=False,
)

# Demo con texto sintético para visualizar el chunking
TEXTO_DEMO = (
    "The Pacific red snapper (Lutjanus peru) is a commercially important demersal fish "
    "species distributed along the eastern Pacific coast from Mexico to Peru. "
    "In the Gulf of California, this species supports one of the most valuable artisanal "
    "fisheries, with annual landings exceeding 8,000 metric tons.\n\n"
    "Population dynamics studies are essential for sustainable fisheries management. "
    "Length-frequency analysis, growth parameters, and mortality rates provide the "
    "biological foundation for stock assessment models such as VPA and CMSY.\n\n"
    "Recent studies using machine learning approaches have demonstrated improved "
    "accuracy in abundance forecasting. Convolutional neural networks trained on "
    "acoustic backscatter data can identify species-specific signatures with 93% accuracy.\n\n"
) * 3  # repetir para tener suficiente texto

chunks = chunker.chunk_text(TEXTO_DEMO, paper_id='demo_paper', source_pdf='demo.pdf')
stats  = chunker.get_stats(chunks)

print(f'Texto original : {len(TEXTO_DEMO):,} caracteres')
print(f'Chunks generados: {stats["total"]}')
print(f'Chars por chunk : min={stats["chars_min"]}  avg={stats["chars_avg"]}  max={stats["chars_max"]}')
print()
for i, c in enumerate(chunks):
    print(f'  chunk_{i:03d}  [{c.char_start:,}–{c.char_end:,}]  {len(c.text)} chars  pág.{c.page_number}')

In [ ]:
# Visualizar el solapamiento entre chunks consecutivos
if len(chunks) >= 2:
    c0, c1 = chunks[0], chunks[1]
    overlap_start = c1.char_start
    overlap_end   = c0.char_end
    overlap_text  = TEXTO_DEMO[overlap_start:overlap_end]

    print('SOLAPAMIENTO entre chunk_000 y chunk_001:')
    print(f'  char_end  del chunk_000 : {c0.char_end}')
    print(f'  char_start del chunk_001 : {c1.char_start}')
    print(f'  Overlap real : {max(0, c0.char_end - c1.char_start)} chars')
    print(f'\n  Texto solapado: "{overlap_text[:150]}..."')

### 3.2 — PdfPlumberExtractor: extracción de texto de PDFs

Los PDFs científicos son complejos: layouts de dos columnas, fórmulas matemáticas, tablas, headers/footers repetitivos.  
`PdfPlumberExtractor` maneja todos estos casos automáticamente.

In [ ]:
from pipeline.rag import PdfPlumberExtractor

# PDFs disponibles en el proyecto
pdf_dir = ROOT / 'outputs' / 'pdfs'
pdfs = sorted(pdf_dir.glob('*.pdf'))

print(f'PDFs encontrados en {pdf_dir.relative_to(ROOT)}:')
for p in pdfs:
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name[:70]}  ({size_kb:.0f} KB)')

In [ ]:
extractor = PdfPlumberExtractor(
    min_page_chars=50,
    strip_headers=True,   # elimina headers/footers repetitivos
    verbose=False,
)

if pdfs:
    pdf_path = pdfs[0]  # primer PDF disponible
    print(f'Extrayendo: {pdf_path.name}\n')

    pages = extractor.extract_by_pages(pdf_path)

    print(f'Páginas extraídas : {len(pages)}')
    total_chars = sum(len(t) for _, t in pages)
    print(f'Total caracteres  : {total_chars:,}')
    print(f'Promedio por pág. : {total_chars // len(pages):,} chars')
    print()

    # Mostrar fragmento de la primera página
    pnum, ptext = pages[0]
    print(f'--- Página {pnum} (primeros 400 chars) ---')
    print(ptext[:400])
else:
    print('No hay PDFs disponibles. Ejecuta primero:')
    print('  python buscar.py "Lutjanus peru" --download')

### 3.3 — VectorDBManager: índice FAISS

**FAISS** (Facebook AI Similarity Search) es una librería de búsqueda de vecinos más cercanos en espacios de alta dimensión.

Usamos `IndexFlatIP` (Inner Product) con vectores normalizados a norma 1, lo que equivale a **similitud coseno**:

```
score = cos(θ) = (v_query · v_chunk) / (|v_query| × |v_chunk|)
         score ∈ [-1, 1]  →  convertido a [0, 1]
```

El índice persiste en 3 archivos:
- `index.faiss`         — vectores binarios
- `metadata_store.json` — metadatos de cada chunk (título, autores, página, etc.)
- `index_config.json`   — modelo, dimensión, timestamp

In [ ]:
import tempfile
from pipeline.rag import VectorDBManager, ChunkData, ChunkVector

# Usamos un directorio temporal para esta demo
# (no modifica el índice real en outputs/rag_index/)
DEMO_INDEX = Path(tempfile.mkdtemp()) / 'demo_index'

db = VectorDBManager(
    index_dir=DEMO_INDEX,
    embedding_dim=gen.get_dimension(),
    index_type='FlatIP',
    verbose=False,
)

print(f'Índice creado en: {DEMO_INDEX}')
print(f'Tipo             : FlatIP (cosine similarity)')
print(f'Dimensión        : {gen.get_dimension()}')

In [ ]:
# Indexar abstracts de los artículos encontrados en la búsqueda
# (flujo alternativo cuando no hay PDFs: title + abstract → chunks)

total_indexed = 0
chunk_vectors_all = []

for i, article in enumerate(articles_with_abstract):
    # Combinar título + abstract como texto fuente
    text = f"{article.title}.\n\n{article.abstract}"
    paper_id = f"art_{i:03d}"

    # Chunkear
    chunks = chunker.chunk_text(text, paper_id=paper_id, source_pdf='')
    if not chunks:
        continue

    # Generar embeddings
    texts_chunk = [c.text for c in chunks]
    vectors = gen.batch_generate(texts_chunk)

    # Enriquecer chunks con metadatos del artículo
    for c in chunks:
        c.title   = article.title
        c.authors = article.authors
        c.year    = article.year
        c.doi     = article.doi

    # Ensamblar ChunkVector y agregar al índice
    cvs = [
        ChunkVector(chunk=c, vector=vectors[j], embedding_model=gen.get_model_name())
        for j, c in enumerate(chunks)
    ]
    db.add_chunks(cvs)
    chunk_vectors_all.extend(cvs)
    total_indexed += len(cvs)

# Guardar índice en disco
db.save()

stats = db.get_stats()
print(f'Chunks indexados  : {stats.total_chunks}')
print(f'Papers en índice  : {stats.total_papers}')
print(f'Modelo            : {stats.embedding_model}')
print(f'Tamaño en disco   : {stats.index_size_mb:.3f} MB')
print(f'\nArchivos generados:')
for f in sorted(DEMO_INDEX.iterdir()):
    print(f'  {f.name:30s}  {f.stat().st_size:,} bytes')

### 3.4 — Búsqueda semántica sobre el índice

Con el índice construido, podemos hacer búsquedas en lenguaje natural.  
El flujo es:

```
pregunta (texto)  →  EmbeddingGenerator  →  query_vector
                                                  ↓
                                      VectorDBManager.search(top_k=5)
                                                  ↓
                                       [RAGSearchResult, ...]  ordenados por score
```

In [ ]:
from pipeline.rag import RAGSearchResult

def buscar(pregunta: str, top_k: int = 5) -> None:
    """Ejecuta una búsqueda semántica e imprime los resultados."""
    print(f'QUERY: "{pregunta}"')
    print('─' * 70)

    # 1. Vectorizar la pregunta
    query_vec = gen.batch_generate([pregunta])[0]

    # 2. Buscar en FAISS
    results = db.search(query_vec, top_k=top_k)

    if not results:
        print('Sin resultados.')
        return

    for r in results:
        authors_str = ', '.join((r.authors or [])[:2])
        if r.authors and len(r.authors) > 2:
            authors_str += ' et al.'
        print(f'[{r.score:.3f}] {r.chunk_id}')
        print(f'         Título  : {r.title or r.paper_id}')
        print(f'         Autores : {authors_str or "N/A"}  ({r.year or "?"})')
        print(f'         Texto   : {r.text[:200]}...' if len(r.text) > 200 else f'         Texto   : {r.text}')
        print()

# Búsqueda 1: población y mortalidad
buscar("population growth mortality rate stock assessment")

In [ ]:
# Búsqueda 2: hábitos alimenticios
buscar("feeding habits prey diet stomach contents")

In [ ]:
# Búsqueda 3: relación talla-peso
buscar("length weight relationship condition factor growth parameters")

### 3.5 — Flujo completo con RAGPipelineOrchestrator (PDFs reales)

`RAGPipelineOrchestrator` conecta todos los componentes en un solo objeto y procesa PDFs completos.  
Este es el flujo que también ejecuta `python indexar.py`.

In [ ]:
from pipeline.rag import RAGPipelineOrchestrator

REAL_INDEX = ROOT / 'outputs' / 'rag_index'

orchestrator = RAGPipelineOrchestrator(
    pdf_dir=pdf_dir,
    index_dir=REAL_INDEX,
    extractor=PdfPlumberExtractor(verbose=False),
    chunker=TextChunker(chunk_size=2000, overlap=200, verbose=False),
    embedding_generator=gen,
    skip_indexed=True,   # omite PDFs ya indexados → idempotente
    batch_size=32,
    verbose=True,
)

print(f'PDFs en {pdf_dir.relative_to(ROOT)}: {len(pdfs)}')
print(f'Índice en: {REAL_INDEX.relative_to(ROOT)}')
print()

In [ ]:
if pdfs:
    run_stats = orchestrator.run()
    print()
    print(f'Procesados  : {run_stats["processed"]}')
    print(f'Omitidos    : {run_stats["skipped"]}  (ya indexados)')
    print(f'Errores     : {len(run_stats["failed"])}')
    print(f'Chunks nuevos: {run_stats["total_chunks"]}')
    print()
    real_stats = run_stats.get('index_stats')
    if real_stats:
        print(f'Índice total : {real_stats.total_chunks} chunks de {real_stats.total_papers} papers')
        print(f'Tamaño       : {real_stats.index_size_mb:.2f} MB')
else:
    print('No hay PDFs. Descarga con:  python buscar.py "Lutjanus peru" --download')

In [ ]:
# Búsqueda semántica sobre el índice real de PDFs
if pdfs and REAL_INDEX.exists():
    import json

    config_path = REAL_INDEX / 'index_config.json'
    with open(config_path) as f:
        cfg = json.load(f)

    real_db = VectorDBManager(
        index_dir=REAL_INDEX,
        embedding_dim=cfg['embedding_dimension'],
    )
    real_db.load()

    print(f'Papers indexados: {real_db.get_papers_indexed()}')
    print()

    def buscar_real(pregunta: str, top_k: int = 3) -> None:
        print(f'QUERY: "{pregunta}"')
        print('─' * 70)
        q = gen.batch_generate([pregunta])[0]
        for r in real_db.search(q, top_k=top_k):
            print(f'  [{r.score:.3f}]  {r.chunk_id}  (pág. {r.page_number})')
            print(f'           {r.text[:180]}...' if len(r.text)>180 else f'           {r.text}')
            print()

    buscar_real("feeding habits prey items stomach content Lutjanus")

In [ ]:
if pdfs and REAL_INDEX.exists():
    buscar_real("growth parameters von Bertalanffy asymptotic length")

---
## Resumen del Pipeline

```
FASE 1  ScientificArticleSearcher
        buscar.py "Lutjanus peru" --sources crossref pubmed arxiv
        → outputs/search_results/*.csv

FASE 2  EmbeddingGenerator (all-MiniLM-L6-v2)
        get_embedding_generator(provider='local')
        .batch_generate(texts) → np.ndarray (N × 384)

FASE 3  RAGPipelineOrchestrator
        PdfPlumberExtractor  → páginas de texto
        TextChunker          → chunks de 2000 chars / 200 overlap
        EmbeddingGenerator   → vectores float32
        VectorDBManager      → FAISS FlatIP + metadata JSON

        indexar.py --pdf-dir outputs/pdfs/ --verbose
        → outputs/rag_index/

FASE 4  RAGQueryEngine  ← PRÓXIMA FASE
        pregunta → vector → top-k chunks → Claude API → respuesta
```

### Comandos rápidos

```bash
# Búsqueda + descarga + indexado en un solo paso
python buscar.py "Lutjanus peru Gulf of California" --download --index

# Solo indexar PDFs existentes
python indexar.py --verbose

# Ver estadísticas del índice
python indexar.py --stats

# Listar papers indexados
python indexar.py --list

# Re-indexar forzando aunque ya existan
python indexar.py --force
```

### Estado del proyecto

| Fase | Descripción | Estado |
|------|-------------|--------|
| 1 | Búsqueda multi-fuente | ✅ Completo |
| 2 | Generación de embeddings | ✅ Completo |
| 3 | RAG pipeline (PDF → índice FAISS) | ✅ Completo |
| 4 | RAG Query Engine (pregunta → respuesta con Claude API) | ⏳ Pendiente |
| 5 | GraphRAG (grafo de entidades y relaciones) | ⏳ Pendiente |